# LogiCrisis: Multi-Agent Logistics Recovery — GRPO Training
**Meta PyTorch OpenEnv Hackathon | Theme #1: Multi-Agent Interactions**

Stack: `Unsloth` (4-bit QLoRA) + `TRL GRPOTrainer` + `LogiCrisisEnv` (OpenEnv)

> Runtime: GPU (T4 or A100 recommended)

In [ ]:
# Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl>=0.8.6 datasets accelerate peft fastapi uvicorn gradio

In [ ]:
# Clone / mount the LogiCrisis environment
# Option A: mount Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# import sys; sys.path.insert(0, '/content/drive/MyDrive/logicriasis')

# Option B: Clone from HuggingFace Space (after openenv push)
# !git clone https://huggingface.co/spaces/<your-username>/logicriasis
# import sys; sys.path.insert(0, '/content/logicriasis')

# For now: paste the environment inline or upload the folder
import sys, os
# sys.path.insert(0, '/content/logicriasis')  # adjust path
print('Path configured')

In [ ]:
# ── Load model with Unsloth ──────────────────────────────────────────────
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/llama-3-8b-instruct-bnb-4bit"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print('Model loaded:', MODEL_NAME)

In [ ]:
# ── Build prompt dataset from environment observations ───────────────────
from training.train import build_prompt_dataset

CURRICULUM_LEVEL = 1   # Start easy!
dataset = build_prompt_dataset(n_samples=64, curriculum_level=CURRICULUM_LEVEL)
print(f'Dataset size: {len(dataset)} prompts')
print('Sample prompt (first 400 chars):')
print(dataset[0]['prompt'][:400])

In [ ]:
# ── GRPO reward function (verifiable, 6 signals) ────────────────────────
from training.train import grpo_reward_fn, _score_completion

# Quick sanity check
test_good = '{"action_type": "reroute", "cargo_id": "C001", "route_id": "Mumbai-Pune", "reasoning": "Rerouting via open road to avoid flood zone"}'
test_bad  = '{"action_type": "INVALID"}'
print('Good action score:', _score_completion(test_good))
print('Bad action score: ', _score_completion(test_bad))

In [ ]:
# ── GRPO Training ────────────────────────────────────────────────────────
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="/content/logicriasis_outputs",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=5,
    save_steps=50,
    report_to="none",
    max_completion_length=256,
    num_generations=8,
    temperature=0.7,
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=grpo_reward_fn,
    args=grpo_config,
    train_dataset=dataset,
)

print('Starting GRPO training…')
trainer.train()

In [ ]:
# ── Save adapters (DO NOT naive-merge 4-bit — guide section 16) ──────────
SAVE_DIR = "/content/logicriasis_outputs/final"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Adapters saved to {SAVE_DIR}')

In [ ]:
# ── Before/after benchmark ───────────────────────────────────────────────
from benchmark import run_benchmark

print('=== BASELINE (heuristic, no training) ===')
baseline = run_benchmark(n_episodes=20, curriculum_level=1)
for k, v in baseline.items():
    print(f'  {k}: {v}')

# After training, re-run with trained model inference
# (plug in the trained model's generate() into run_heuristic_episode)
print('\nTrain the model above, then compare!')

In [ ]:
# ── Launch Gradio demo ───────────────────────────────────────────────────
import subprocess
subprocess.Popen(['python', 'demo/app.py'])
print('Demo running on port 7860')